In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import mean_squared_error, r2_score, f1_score, roc_auc_score, confusion_matrix, precision_recall_curve

df = pd.read_csv('creditcard.csv')

fraud = df[df['Class'] == 1]
legit = df[df['Class'] == 0].sample(n=len(fraud)*3, random_state=1984)  
df_bal = pd.concat([fraud, legit]).sample(frac=1, random_state=1984).reset_index(drop=True)
X_clf = df_bal.drop(['Time', 'Class'], axis=1).values
y_clf = df_bal['Class'].values

print('подготовка данных')
print(f'всего строк: {len(df_bal)}')
print(f'фрод: {y_clf.sum()}, норма: {(y_clf==0).sum()}')

df_legit = df[df['Class'] == 0]
feature_names = [c for c in df_legit.columns if c not in ['Amount', 'Class']]
X_reg = df_legit.drop(['Amount', 'Class'], axis=1).values
y_reg = df_legit['Amount'].values

X_tr, X_te, y_tr, y_te = train_test_split(X_reg, y_reg, test_size=0.2, random_state=1984)
sc_r = StandardScaler()
X_tr_sc, X_te_sc = sc_r.fit_transform(X_tr), sc_r.transform(X_te)

reg_mdls = {'linear': LinearRegression(), 'ridge': Ridge(), 'lasso': Lasso(alpha=0.1), 
            'rf': RandomForestRegressor(n_estimators=20, max_depth=10, random_state=1984, n_jobs=-1)}
reg_res = {}
for name, mdl in reg_mdls.items():
    X_train, X_test = (X_tr_sc, X_te_sc) if name != 'rf' else (X_tr, X_te)
    mdl.fit(X_train, y_tr)
    pred = mdl.predict(X_test)
    rmse_val = np.sqrt(mean_squared_error(y_te, pred))
    r2_val = r2_score(y_te, pred)
    
    if name == 'rf':
        importance = mdl.feature_importances_
    else:
        importance = np.abs(mdl.coef_)
    top_idx = np.argsort(importance)[::-1][:5]
    top_feats = [feature_names[i] for i in top_idx]
    
    reg_res[name] = {'rmse': rmse_val, 'r2': r2_val, 'top_features': top_feats}

print('\nрегрессия')
for name, res in reg_res.items():
    feats = ', '.join(res['top_features'])
    print(f'{name}: RMSE = {res["rmse"]:.2f}, R2 = {res["r2"]:.4f}')
    print(f'топ-5 признаков: {feats}')

best_reg = max(reg_res, key=lambda k: reg_res[k]['r2'])
print(f'\nлучшая модель: {best_reg}')
print(f'причина: максимальный R2 = {reg_res[best_reg]["r2"]:.4f}')

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=1984)
clf_mdls = {
    'logreg': Pipeline([('sc', StandardScaler()), ('clf', LogisticRegression(max_iter=1000, random_state=1984))]),
    'rf': RandomForestClassifier(n_estimators=50, max_depth=6, random_state=1984, n_jobs=-1)
}

clf_res = {}
for name, mdl in clf_mdls.items():
    y_prob = cross_val_predict(mdl, X_clf, y_clf, cv=cv, method='predict_proba')[:, 1]
    
    prec, rec, thresh = precision_recall_curve(y_clf, y_prob)
    f1s = 2 * prec * rec / (prec + rec + 1e-8)
    best_idx = np.argmax(f1s[:-1])  
    best_thresh = thresh[best_idx]
    best_f1 = f1s[best_idx]
    
    y_pred = (y_prob >= best_thresh).astype(int)
    auc_val = roc_auc_score(y_clf, y_prob)
    clf_res[name] = {'y_prob': y_prob, 'y_pred': y_pred, 'thresh': best_thresh, 'f1': best_f1, 'auc': auc_val}

print('\nклассификация')
for name, res in clf_res.items():
    print(f'{name}: порог = {res["thresh"]:.3f}, F1 = {res["f1"]:.4f}, AUC-ROC = {res["auc"]:.4f}')

best_clf = max(clf_res, key=lambda k: clf_res[k]['f1'])
print(f'\nлучшая модель: {best_clf}')
print(f'причина: максимальный F1 = {clf_res[best_clf]["f1"]:.4f}')

cm = confusion_matrix(y_clf, clf_res[best_clf]['y_pred'])
thresh_val = clf_res[best_clf]['thresh']

print(f'\nматрица ошибок (лучшая модель: {best_clf}, порог: {thresh_val:.2f})')
print(f'true negatives: {cm[0,0]}', f'false positives: {cm[0,1]}', f'false negatives: {cm[1,0]}', f'true positives: {cm[1,1]}', sep='\n')

подготовка данных
всего строк: 1968
фрод: 492, норма: 1476



регрессия
linear: RMSE = 76.00, R2 = 0.9058
топ-5 признаков: V2, V7, V5, V20, V1
ridge: RMSE = 76.00, R2 = 0.9058
топ-5 признаков: V2, V7, V5, V20, V1
lasso: RMSE = 75.99, R2 = 0.9058
топ-5 признаков: V2, V7, V5, V20, V1
rf: RMSE = 57.23, R2 = 0.9466
топ-5 признаков: V7, V2, V20, V5, V10

лучшая модель: rf
причина: максимальный R2 = 0.9466



классификация
logreg: порог = 0.354, F1 = 0.9237, AUC-ROC = 0.9776
rf: порог = 0.469, F1 = 0.9186, AUC-ROC = 0.9727

лучшая модель: logreg
причина: максимальный F1 = 0.9237

матрица ошибок (лучшая модель: logreg, порог: 0.35)
true negatives: 1453
false positives: 23
false negatives: 50
true positives: 442
